# CenterNet（PASCAL VOC 2007を用いた物体検出）

---
## 目的
物体を「矩形の中心点」として捉えるkeypointベースの物体検出手法CenterNet（Objects as Points）を，PASCAL VOC 2007データセットを用いて実装する．CenterNetは，anchor（デフォルトボックス）を一切使わず，各クラスの中心点ヒートマップを予測するだけで物体検出を行う，非常にシンプルなパイプラインを持つ．SSDのような複雑なデフォルトボックス設計や，CornerNetのようなコーナーのペアリング（embedding）も不要になる仕組みを理解する．

## モジュールのインポート
`torchmetrics`は標準ではインストールされていないため，はじめに`pip install`でインストールしたのち，必要なモジュールをインポートし，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
!pip install -q torchmetrics

import os
import zipfile
from time import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.datasets import VOCDetection
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.ops import nms
from torchvision.utils import draw_bounding_boxes
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
PASCAL VOC 2007データセットを読み込みます．VOCデータセットは，人物・乗り物・動物・室内物品など20クラスの物体に対して矩形領域（Bounding Box）が付与されたデータセットで，学習用（trainval）5011枚，評価用（test）4952枚の画像から構成されます．データセットの読み込みやDataLoaderの基本的な使い方については`02_dnn_simple_pytorch/existing_dataset.ipynb`を参照してください．

ここでは，Googleドライブに配置したVOCdevkit（2007）のzipファイルを`gdown`でダウンロードし，`torchvision.datasets.VOCDetection`で読み込みます．CenterNetはバックボーンで特徴マップを1/4サイズまでダウンサンプリングしたのち，各種ヘッドを適用するため，入力画像サイズ`IMG_SIZE`は4で割り切れる256とします．

In [ ]:
VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
# 背景（物体なし）を0番として，クラスラベルは1〜20を割り当てる
CLASS_TO_IDX = {name: i + 1 for i, name in enumerate(VOC_CLASSES)}
n_class = len(VOC_CLASSES) + 1  # 背景を含めたクラス数（データセット側のラベル付けはSSDと共通にする）
IMG_SIZE = 256

voc_root = './'
devkit_dir = os.path.join(voc_root, 'VOCdevkit')
if not os.path.isdir(devkit_dir):
    gdown.download(id='1xOO8ViO8NPyPngm24POj5a49zZOSkkMg', output='VOCdevkit.zip', quiet=False)
    with zipfile.ZipFile('VOCdevkit.zip') as f:
        f.extractall(voc_root)


def parse_voc_target(target):
    """VOCDetectionが返すXML由来のdictを，(boxes, labels)のTensorに変換する"""
    objects = target['annotation']['object']
    if isinstance(objects, dict):  # 物体が1つだけの場合はdictのまま返るため，リストに揃える
        objects = [objects]

    boxes, labels = [], []
    for obj in objects:
        bbox = obj['bndbox']
        boxes.append([float(bbox['xmin']), float(bbox['ymin']), float(bbox['xmax']), float(bbox['ymax'])])
        labels.append(CLASS_TO_IDX[obj['name']])

    return {'boxes': torch.tensor(boxes, dtype=torch.float32), 'labels': torch.tensor(labels, dtype=torch.long)}


class VOCDetectionDataset(torch.utils.data.Dataset):
    """VOCDetectionをラップし，画像のリサイズに合わせてボックス座標をスケールしたTensorを返すDataset"""
    def __init__(self, image_set, img_size=IMG_SIZE):
        self.voc = VOCDetection(root=voc_root, year='2007', image_set=image_set, download=False)
        self.img_size = img_size
        self.to_tensor = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.voc)

    def __getitem__(self, idx):
        image, target = self.voc[idx]
        w, h = image.size
        target = parse_voc_target(target)

        # 元画像サイズ -> (img_size, img_size) へのリサイズに合わせてボックス座標をスケール
        scale = torch.tensor([self.img_size / w, self.img_size / h, self.img_size / w, self.img_size / h])
        target['boxes'] = target['boxes'] * scale

        image = self.to_tensor(image)
        return image, target


def collate_fn(batch):
    # 画像1枚あたりの物体数が異なるため，ボックス・ラベルはリストのまま扱う
    images = torch.stack([item[0] for item in batch])
    targets = [item[1] for item in batch]
    return images, targets


train_data = VOCDetectionDataset('trainval')
test_data = VOCDetectionDataset('test')

print(len(train_data), len(test_data))
print('クラス数（背景含む）:', n_class)

## CenterNet（Objects as Points）
CenterNetは，物体を「矩形の中心点」という単一のキーポイントとして検出する手法です．SSDのように大量のデフォルトボックスを画像全体に敷き詰めて分類・回帰する必要も，CornerNetのように2つのコーナー（左上・右下）を検出してペアリング（embedding）する必要もありません．クラスごとの中心点ヒートマップ上でピークを探すだけで，物体の位置とクラスが同時に得られます．

### バックボーンとアップサンプリング
CenterNetの原論文では，Hourglass Network・DLA・ResNet+Deconvなど複数のバックボーン構成が示されていますが，本ノートブックでは事前学習済みのResNet50に，原論文の「ResNet+Deconv」構成と同じ発想の，3段のTransposed Convolution（`nn.ConvTranspose2d`）によるアップサンプリング層を組み合わせます．ResNet50は入力に対してstride 32まで特徴マップを縮小しますが，stride 2のアップサンプリングを3回繰り返すことで，ちょうどstride 4（`IMG_SIZE=256`に対して64×64）まで解像度を戻します．このstride 4の特徴マップ上で，後述する各ヘッドを適用します．

In [ ]:
OUTPUT_STRIDE = 4
FEAT_SIZE = IMG_SIZE // OUTPUT_STRIDE  # ヒートマップ・各ヘッドの出力解像度


def deconv_block(in_channels, out_channels):
    # kernel=4, stride=2, padding=1 とすることで，特徴マップの解像度がちょうど2倍になる
    return nn.Sequential(
        nn.ConvTranspose2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
    )


class CenterNetBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.stem = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4,
        )  # stride 32, channels 2048

        # stride 32 -> 16 -> 8 -> 4 の順にアップサンプリングし，解像度を取り戻す
        self.up1 = deconv_block(2048, 256)
        self.up2 = deconv_block(256, 128)
        self.up3 = deconv_block(128, 64)
        self.out_channels = 64

    def forward(self, x):
        x = self.stem(x)
        x = self.up1(x)
        x = self.up2(x)
        x = self.up3(x)
        return x  # (B, 64, FEAT_SIZE, FEAT_SIZE)

### 検出ヘッド（ヒートマップ・サイズ・オフセット）
バックボーンが出力するstride 4の特徴マップに対して，3つの小さな畳み込みヘッドを並列に適用します．いずれも「3×3畳み込み + ReLU + 1×1畳み込み」という共通の構造を持つため，`CenterNetHead`として1つのクラスにまとめて実装します．

1. **ヒートマップヘッド**（出力チャンネル数20 = クラス数のみ．背景チャンネルは持たない）：各クラス・各画素が「そのクラスの物体の中心である確率」をSigmoidで出力します．SSDのようにデフォルトボックスごとに背景クラスを明示的に持たせる必要がなく，各クラスのヒートマップ値がそのまま信頼度になります．
2. **サイズ（w, h）ヘッド**（出力チャンネル数2）：中心点の位置においてのみ，ボックスの幅・高さをstride 4の特徴マップ座標系のスケールで直接回帰します．SSDのようにデフォルトボックスからの相対オフセット（対数スケール）にエンコードするのではなく，生の値をそのまま回帰する点がCenterNetの大きな単純化の1つです．
3. **オフセットヘッド**（出力チャンネル数2）：中心点の座標をstride 4でダウンサンプリングした際に生じる整数化誤差（量子化誤差）を補正するための，サブピクセル単位の(x, y)オフセットを回帰します．

ヒートマップヘッドの最終層のバイアスは，学習初期に出力が0（背景）に極端に偏るのを防ぐため，Sigmoid後の出力がおよそ0.1になるよう`-2.19`で初期化します（RetinaNetやCornerNetでも使われる定番のテクニックです）．

In [ ]:
class CenterNetHead(nn.Module):
    def __init__(self, in_channels, out_channels, bias_fill=None):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, out_channels, kernel_size=1),
        )
        if bias_fill is not None:
            nn.init.constant_(self.net[-1].bias, bias_fill)

    def forward(self, x):
        return self.net(x)

### Ground Truthヒートマップ・回帰ターゲットの生成
学習時には，各GTボックスの中心座標をstride 4の特徴マップ座標系に変換し，そのクラスのヒートマップ上にGaussian分布を書き込みます（中心そのものだけを正例の1点とするのではなく，周辺の画素にもなだらかに正例らしさを与えることで，学習を安定させます）．これは，2つのコーナーそれぞれにGaussianターゲットを置くCornerNetのヒートマップ生成と同じ考え方で，CenterNetでは対象がボックス中心1点だけになるため，コーナーのペアリング（embedding）が不要になります．

Gaussianの半径は，GTボックスのサイズから，生成したボックスと元のGTボックスとのIoUが一定値（`min_overlap`）以上になるように，CornerNet/CenterNetで標準的に使われる二次方程式の解の公式から計算します．複数のGTボックスのGaussianが重なる場合は，加算せず要素ごとの最大値（`torch.max`）を取ることで，ヒートマップの値が1を超えないようにします．

サイズ・オフセットの回帰ターゲットは，中心点の画素位置にのみ値を持つ密なマップとして作成し，どの画素が正例（GTの中心）かを`mask`で管理します．損失計算では，この`mask`を使って中心点以外の画素を無視します．

In [ ]:
def gaussian_radius(height, width, min_overlap=0.7):
    """GTボックスと生成ボックスのIoUがmin_overlap以上になるGaussian半径を求める（CornerNet/CenterNetで標準的に使われる式，height/widthはPythonのfloat）"""
    a1 = 1
    b1 = height + width
    c1 = width * height * (1 - min_overlap) / (1 + min_overlap)
    r1 = (b1 + max(b1 ** 2 - 4 * a1 * c1, 0) ** 0.5) / 2

    a2 = 4
    b2 = 2 * (height + width)
    c2 = (1 - min_overlap) * width * height
    r2 = (b2 + max(b2 ** 2 - 4 * a2 * c2, 0) ** 0.5) / 2

    a3 = 4 * min_overlap
    b3 = -2 * min_overlap * (height + width)
    c3 = (min_overlap - 1) * width * height
    r3 = (b3 + max(b3 ** 2 - 4 * a3 * c3, 0) ** 0.5) / 2

    return min(r1, r2, r3)


def gaussian_2d(radius, sigma_scale=3.0):
    """半径radiusの正方形カーネルとして，2次元Gaussian分布を生成する"""
    sigma = max(radius / sigma_scale, 1e-3)
    coords = torch.arange(-radius, radius + 1, dtype=torch.float32)
    yy, xx = torch.meshgrid(coords, coords, indexing='ij')
    gauss = torch.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    gauss[gauss < torch.finfo(gauss.dtype).eps * gauss.max()] = 0
    return gauss


def draw_gaussian(heatmap, center_x, center_y, radius):
    """heatmap（1クラス分，2次元）上の(center_x, center_y)にGaussianを書き込む（重なりは要素ごとの最大値を採用）"""
    diameter = 2 * radius + 1
    gaussian = gaussian_2d(radius).to(heatmap.device)
    height, width = heatmap.shape

    left, right = min(center_x, radius), min(width - center_x, radius + 1)
    top, bottom = min(center_y, radius), min(height - center_y, radius + 1)

    masked_heatmap = heatmap[center_y - top:center_y + bottom, center_x - left:center_x + right]
    masked_gaussian = gaussian[radius - top:radius + bottom, radius - left:radius + right]
    if min(masked_heatmap.shape) > 0 and min(masked_gaussian.shape) > 0:
        torch.max(masked_heatmap, masked_gaussian, out=masked_heatmap)
    return heatmap


def generate_targets(gt_boxes, gt_labels, feat_size=FEAT_SIZE, stride=OUTPUT_STRIDE, num_classes=n_class - 1):
    """1枚の画像内のGTボックスから，ヒートマップ・サイズ・オフセットの学習ターゲットを作成する"""
    heatmap = torch.zeros(num_classes, feat_size, feat_size, device=device)
    wh_target = torch.zeros(2, feat_size, feat_size, device=device)
    offset_target = torch.zeros(2, feat_size, feat_size, device=device)
    mask = torch.zeros(feat_size, feat_size, device=device)

    for box, label in zip(gt_boxes, gt_labels):
        x1, y1, x2, y2 = (box / stride).tolist()  # 特徴マップ座標系（stride 4）に変換
        w, h = x2 - x1, y2 - y1
        if w <= 0 or h <= 0:
            continue
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        cx_int, cy_int = int(cx), int(cy)
        if not (0 <= cx_int < feat_size and 0 <= cy_int < feat_size):
            continue

        radius = max(0, int(gaussian_radius(h, w)))
        class_idx = label.item() - 1  # 背景を除いた0-indexedのクラスID（ヒートマップのチャンネルに対応）
        draw_gaussian(heatmap[class_idx], cx_int, cy_int, radius)

        # サイズ・オフセットは中心点の画素にのみ値を書き込む（それ以外はmask=0として損失計算時に無視する）
        wh_target[:, cy_int, cx_int] = torch.tensor([w, h], device=device)
        offset_target[:, cy_int, cx_int] = torch.tensor([cx - cx_int, cy - cy_int], device=device)
        mask[cy_int, cx_int] = 1.0

    return heatmap, wh_target, offset_target, mask

## 損失関数
CenterNetの損失は，3つの項の和で構成されます．

- **ヒートマップ損失**：Penalty-reduced Focal Loss．通常のFocal Lossに加え，正例（GTの中心）からの距離が近い負例画素ほどペナルティを弱める`(1 - target)^β`という重みを掛けます．これは，Gaussianの裾野に位置する「中心に近いがズレている」画素を，遠く離れた背景画素と同程度に強く負例として罰してしまうと学習が不安定になるためで，CornerNet・CenterNetに共通する特徴的な工夫です（通常のRetinaNetのFocal Lossにはこの重みはありません）．
- **サイズ損失**：GTの中心点における(w, h)予測に対するL1損失．
- **オフセット損失**：GTの中心点における(x, y)オフセット予測に対するL1損失．

サイズ損失は，値のスケールがヒートマップ・オフセット損失より大きくなりやすいため，原論文に従って重み0.1を掛けて合算します．

In [ ]:
class CenterNetLoss(nn.Module):
    def __init__(self, alpha=2, beta=4, size_weight=0.1, offset_weight=1.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.size_weight = size_weight
        self.offset_weight = offset_weight

    def focal_loss(self, pred, target):
        pred = pred.clamp(1e-4, 1 - 1e-4)
        pos_mask = target.eq(1).float()  # Gaussianのピーク（GT中心そのもの）のみが正例
        neg_mask = target.lt(1).float()
        neg_weight = torch.pow(1 - target, self.beta)  # 中心に近い負例ほどペナルティを弱める

        pos_loss = torch.log(pred) * torch.pow(1 - pred, self.alpha) * pos_mask
        neg_loss = torch.log(1 - pred) * torch.pow(pred, self.alpha) * neg_weight * neg_mask

        num_pos = pos_mask.sum().clamp(min=1)
        return -(pos_loss.sum() + neg_loss.sum()) / num_pos

    def reg_l1_loss(self, pred, target, mask):
        mask = mask.unsqueeze(1)  # (B, H, W) -> (B, 1, H, W)，(B, 2, H, W)にブロードキャストする
        loss = F.l1_loss(pred * mask, target * mask, reduction='sum')
        return loss / mask.sum().clamp(min=1)

    def forward(self, heatmap_pred, wh_pred, offset_pred, heatmap_target, wh_target, offset_target, mask):
        loss_hm = self.focal_loss(heatmap_pred, heatmap_target)
        loss_wh = self.reg_l1_loss(wh_pred, wh_target, mask)
        loss_off = self.reg_l1_loss(offset_pred, offset_target, mask)
        loss = loss_hm + self.size_weight * loss_wh + self.offset_weight * loss_off
        return loss, loss_hm, loss_wh, loss_off

## ネットワークの作成
バックボーンと3つのヘッドを組み合わせて`CenterNet`モデルを構築します．`.to(device)`によりモデルのパラメータを`device`上に配置します．

最適化手法には，モーメンタムSGDではなくAdam（学習率1.25e-4）を用います．ヒートマップに対するFocal Lossは，SSDのクロスエントロピー損失や物体検出の回帰損失と比べて勾配のスケールが不安定になりやすく，経験的にAdamの方が収束が安定するためです（原論文の公式実装でもAdamが採用されています）．

In [ ]:
class CenterNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = CenterNetBackbone()
        feat_channels = self.backbone.out_channels
        self.heatmap_head = CenterNetHead(feat_channels, num_classes, bias_fill=-2.19)
        self.size_head = CenterNetHead(feat_channels, 2)
        self.offset_head = CenterNetHead(feat_channels, 2)

    def forward(self, x):
        feat = self.backbone(x)
        heatmap = torch.sigmoid(self.heatmap_head(feat))
        wh = self.size_head(feat)
        offset = self.offset_head(feat)
        return heatmap, wh, offset


model = CenterNet(num_classes=n_class - 1)  # 背景クラスを持たないため，クラス数はn_class - 1
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1.25e-4)
criterion = CenterNetLoss(size_weight=0.1, offset_weight=1.0)

n_params = sum(p.numel() for p in model.parameters())
print('総パラメータ数:', n_params)

## 学習
読み込んだVOC2007データセットと作成したネットワークを用いて学習を行います．ミニバッチサイズを8，学習エポック数を20とします．画像1枚あたりの物体数が異なるため，`collate_fn`でバッチを作成し，各画像のGTボックスをバッチ内でループしながらヒートマップ・サイズ・オフセットのターゲットを作成します．SSDのデフォルトボックスマッチングと異なり，`generate_targets`が返すターゲットは画像内の物体数によらず常に同じ形状（`FEAT_SIZE`×`FEAT_SIZE`）になるため，`torch.stack`でそのままバッチ化できます．

In [ ]:
batch_size = 8
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2, collate_fn=collate_fn)

model.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss, sum_loss_hm, sum_loss_wh, sum_loss_off = 0.0, 0.0, 0.0, 0.0

    for images, targets in train_loader:
        images = images.to(device)

        batch_heatmaps, batch_wh, batch_offset, batch_mask = [], [], [], []
        for t in targets:
            hm, wh, off, mask = generate_targets(t['boxes'].to(device), t['labels'].to(device))
            batch_heatmaps.append(hm)
            batch_wh.append(wh)
            batch_offset.append(off)
            batch_mask.append(mask)
        batch_heatmaps = torch.stack(batch_heatmaps)
        batch_wh = torch.stack(batch_wh)
        batch_offset = torch.stack(batch_offset)
        batch_mask = torch.stack(batch_mask)

        heatmap_pred, wh_pred, offset_pred = model(images)

        loss, loss_hm, loss_wh, loss_off = criterion(
            heatmap_pred, wh_pred, offset_pred, batch_heatmaps, batch_wh, batch_offset, batch_mask)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        sum_loss_hm += loss_hm.item()
        sum_loss_wh += loss_wh.item()
        sum_loss_off += loss_off.item()

    n_batch = len(train_loader)
    print("epoch: {}, loss: {:.4f} (heatmap: {:.4f}, size: {:.4f}, offset: {:.4f}), elapsed time: {}".format(
        epoch, sum_loss / n_batch, sum_loss_hm / n_batch, sum_loss_wh / n_batch, sum_loss_off / n_batch,
        time() - train_start))

## 推論（ピーク検出・ボックス復元）
学習したモデルで推論を行うための関数を定義します．CenterNetの推論はSSDよりも大幅にシンプルです．

1. ヒートマップに対して3×3のMax Pooling（`stride=1`）を適用し，各画素の値が自分の3×3近傍の最大値と一致する画素だけを「局所ピーク」として残します．これにより，周辺に埋もれた重複検出が自然に抑制され，SSDのクラスごとのNMSに相当する処理が，ヒートマップ上のピーク検出だけでほぼ完結します．
2. スコアが閾値`score_threshold`を超えるピークのみを残し，スコア上位`max_dets`件に絞ります．
3. 各ピーク画素の位置にオフセットヘッドの出力を加えてサブピクセル精度の中心座標を復元し，サイズヘッドの出力（w, h）から矩形を復元します．最後にstride 4を掛けて元の`IMG_SIZE`のピクセル座標に戻します．
4. 仕上げとして，`torchvision.ops.nms`による軽い重複除去も追加します（ヒートマップのピーク検出自体がNMSに近い役割を果たすため必須ではありませんが，同一物体に対する近接ピークの重複を最終的に取り除く簡単な後処理として加えます）．

In [ ]:
def predict(model, image_tensor, score_threshold=0.3, max_dets=100, nms_threshold=0.5):
    model.eval()
    with torch.no_grad():
        heatmap, wh, offset = model(image_tensor.unsqueeze(0).to(device))
    heatmap, wh, offset = heatmap[0], wh[0], offset[0]  # (C, H, W), (2, H, W), (2, H, W)

    # 3x3のMax Poolingと一致する画素だけを局所ピークとして残す（ヒートマップ版NMS）
    hmax = F.max_pool2d(heatmap, kernel_size=3, stride=1, padding=1)
    peak_mask = (heatmap == hmax) & (heatmap > score_threshold)

    idx = peak_mask.nonzero(as_tuple=False)  # (N, 3): (class_idx, y, x)
    if idx.shape[0] == 0:
        return torch.zeros(0, 4), torch.zeros(0), torch.zeros(0, dtype=torch.long)

    classes, ys, xs = idx[:, 0], idx[:, 1], idx[:, 2]
    scores = heatmap[classes, ys, xs]

    if scores.shape[0] > max_dets:
        scores, top_idx = scores.topk(max_dets)
        classes, ys, xs = classes[top_idx], ys[top_idx], xs[top_idx]

    off = offset[:, ys, xs].t()  # (N, 2)
    size = wh[:, ys, xs].t()  # (N, 2)

    cx = xs.float() + off[:, 0]
    cy = ys.float() + off[:, 1]
    # サイズヘッドの出力に正値制約はないため，学習初期など負の値が出た場合に備えてクリップする（物理的に負のサイズはあり得ない）
    w, h = size[:, 0].clamp(min=0), size[:, 1].clamp(min=0)

    boxes = torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], dim=1) * OUTPUT_STRIDE
    boxes = boxes.clamp(min=0, max=IMG_SIZE)
    labels = classes + 1  # 0-indexedのクラスID -> VOCラベル(1〜20)

    # 軽い後処理としての最終NMS
    keep = nms(boxes, scores, nms_threshold)
    return boxes[keep].cpu(), scores[keep].cpu(), labels[keep].cpu()

## テスト
評価指標には，物体検出タスクで広く使われる**mAP（mean Average Precision）**を用います．mAPの算出は`torchmetrics`の`MeanAveragePrecision`を利用します（詳細は`ssd.ipynb`を参照）．VOCで伝統的に使われるIoU閾値0.5でのmAP（`map_50`）を確認します。

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=1, shuffle=False, collate_fn=collate_fn)

metric = MeanAveragePrecision(iou_type='bbox')
model.eval()

for images, targets in test_loader:
    boxes, scores, labels = predict(model, images[0])

    preds = [{'boxes': boxes, 'scores': scores, 'labels': labels}]
    gts = [{'boxes': targets[0]['boxes'], 'labels': targets[0]['labels']}]
    metric.update(preds, gts)

result = metric.compute()
print('mAP@0.5:', result['map_50'].item())
print('mAP@[0.5:0.95]:', result['map'].item())

## 検出結果の可視化
テストデータ1枚に対する検出結果を，`torchvision.utils.draw_bounding_boxes`を用いて画像上に描画します。

In [ ]:
image, target = test_data[0]
boxes, scores, labels = predict(model, image)

# 正規化を戻して0-255のuint8画像に変換
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
img_show = (image * std + mean).clamp(0, 1)
img_show = (img_show * 255).to(torch.uint8)

label_names = [f'{VOC_CLASSES[l - 1]}: {s:.2f}' for l, s in zip(labels, scores)]
img_with_boxes = draw_bounding_boxes(img_show, boxes, labels=label_names, colors='red', width=2)

plt.figure(figsize=(6, 6))
plt.imshow(img_with_boxes.permute(1, 2, 0))
plt.axis('off')
plt.show()

## オリジナル論文との違い
本ノートブックで実装したCenterNetは，原論文の構成を一部簡略化・変更しています．主な違いは次の通りです．

| | 原論文（CenterNet） | 本ノートブック |
|---|---|---|
| バックボーン | Hourglass-104 / DLA-34 / ResNet+Deconv（複数の構成を報告） | ResNet50（ImageNet事前学習済み）+ Deconv×3（原論文の「ResNet+Deconv」構成に相当） |
| 入力画像サイズ | 512×512 | 256×256 |
| 出力ストライド | 4（共通） | 4（変更なし） |
| Gaussian半径の算出 | IoU下限からの二次方程式の解 | 同じ式を使用（変更なし） |
| 最終後処理 | ヒートマップのピーク検出のみ（NMSなし） | ピーク検出に加え，`torchvision.ops.nms`による軽いNMSを追加 |
| IoU計算・mAP評価 | 独自実装 | `torchvision.ops`・`torchmetrics`を使用 |

損失関数（Penalty-reduced Focal Loss + L1）やヒートマップ・サイズ・オフセットという3つのヘッド構成自体は，原論文と同じ設計です。

## 課題

1. `gaussian_radius`の`min_overlap`や，損失関数の`size_weight`（サイズ損失の重み）を変更し，学習の収束や検出結果への影響を確認しましょう．

2. `predict`関数の中で，学習済みモデルが出力するヒートマップ（`heatmap`）自体を`plt.imshow`で可視化し，物体の中心付近にどのようにGaussian状のピークが立っているかを確認しましょう．

3. 本ノートブックの推論パイプライン（ヒートマップのピーク検出のみ，クラスごとのループが不要）と，`ssd.ipynb`の推論パイプライン（クラスごとにスコアで足切りしたのちNMSを適用）を比較し，処理のシンプルさや推論時間の違いを確認しましょう．